# wav2vec2 Embedding Extraction for Child Vocalisation Classification

This notebook extracts **wav2vec2-base** pretrained embeddings from the denoised audio files.

**Why**: Our from-scratch Transformer models (mel spectrograms, MFCCs) all land at ~0.24 test macro-F1,
indistinguishable from chance. The bottleneck is inter-child variance — the models can't learn
speaker-invariant features from 569 training examples. wav2vec2 has already learned such features
from 960 hours of LibriSpeech.

**Output**: One `.npy` file per clip, shape `(768, MAX_LEN)`. Drop-in compatible with the existing
pipeline on RenkuLab via `USE_PRETRAINED = True` in `config.py`.


## 1. Setup

In [5]:
import torch
import torchaudio
import numpy as np
import os
import re
from pathlib import Path
from google.colab import files
import zipfile
import shutil

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')
print(f'torchaudio: {torchaudio.__version__}')

Device: cpu
PyTorch: 2.10.0+cpu
torchaudio: 2.10.0+cpu


## 2. Load wav2vec2-base pretrained model

In [6]:
# Load wav2vec2-base (95M params, trained on 960h LibriSpeech)
# Output: 768-dim embeddings per frame, one frame per 20ms
bundle = torchaudio.pipelines.WAV2VEC2_BASE
MODEL_SAMPLE_RATE = bundle.sample_rate  # 16000 Hz
EMBEDDING_DIM = 768

model = bundle.get_model().to(device)
model.eval()
print(f'Model loaded on {device}')
print(f'Expected sample rate: {MODEL_SAMPLE_RATE} Hz')
print(f'Embedding dimension: {EMBEDDING_DIM}')

Model loaded on cpu
Expected sample rate: 16000 Hz
Embedding dimension: 768


## 3. Upload audio files

Upload a **zip file** containing your `denoised_audios/` folder.

The zip should contain `.wav` files like:
```
denoised_audios/
  audio_0000.wav
  audio_0001.wav
  ...
  audio_1008.wav
```

Or just the `.wav` files directly at the root of the zip.

In [10]:
uploaded = files.upload()

if not uploaded:
    print('No file uploaded. Please upload a zip file.')
else:
    zip_name = list(uploaded.keys())[0]
    print(f'Uploaded: {zip_name} ({len(uploaded[zip_name]) / 1e6:.1f} MB)')

Saving denoised_audios.zip to denoised_audios.zip
Uploaded: denoised_audios.zip (87.5 MB)


In [13]:
# Extract zip and find wav files
EXTRACT_DIR = Path('/content/audio_files')
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True)

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(EXTRACT_DIR)

# Find all .wav files and filter out macOS hidden files (starting with ._)
wav_files = sorted(
    [f for f in EXTRACT_DIR.rglob('*.wav') if not f.name.startswith('._')],
    key=lambda p: p.name
)

print(f'Found {len(wav_files)} actual .wav files')

if len(wav_files) > 0:
    print(f'First: {wav_files[0].name}')
    print(f'Last:  {wav_files[-1].name}')
else:
    print('ERROR: No valid .wav files found! Check your zip structure.')

Found 1009 actual .wav files
First: audio_0000.wav
Last:  audio_1008.wav


## 4. Extract wav2vec2 embeddings

For each clip:
1. Load audio, resample to 16kHz if needed
2. Forward pass through wav2vec2-base (frozen, no gradients)
3. Take the **last hidden layer** output → (T, 768)
4. Pad or truncate time axis to `MAX_LEN` frames
5. Save as `(768, MAX_LEN)` .npy file

At 16kHz with wav2vec2's 20ms stride:
- 1s audio → ~50 frames
- Mean clip (1.33s) → ~67 frames
- Max clip (8s) → ~400 frames

We use `MAX_LEN=230` to match the existing mel spectrogram pipeline.

In [14]:
MAX_LEN = 230  # Match existing pipeline
OUTPUT_DIR = Path('/content/pretrained_embeddings')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def extract_embedding(wav_path, model, max_len=230):
    """
    Extract wav2vec2-base last-layer embedding from one .wav file.
    Returns (EMBEDDING_DIM, max_len) float32 array.
    """
    waveform, sr = torchaudio.load(wav_path)

    # Resample to 16kHz if needed
    if sr != MODEL_SAMPLE_RATE:
        waveform = torchaudio.functional.resample(waveform, sr, MODEL_SAMPLE_RATE)

    # Convert to mono if stereo
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    waveform = waveform.to(device)

    with torch.no_grad():
        # extract_features returns a list of layer outputs
        features, _ = model.extract_features(waveform)
        # Take last transformer layer (index -1)
        emb = features[-1].squeeze(0)  # (T, 768)

    emb = emb.cpu().numpy()  # (T, 768)

    # Transpose to (768, T) to match mel spectrogram convention
    emb = emb.T  # (768, T)

    # Pad or truncate time axis to max_len
    T = emb.shape[1]
    if T < max_len:
        emb = np.pad(emb, ((0, 0), (0, max_len - T)))
    else:
        emb = emb[:, :max_len]

    return emb.astype(np.float32)  # (768, max_len)


# Process all clips
print(f'Extracting wav2vec2-base embeddings...')
print(f'Output shape per clip: ({EMBEDDING_DIM}, {MAX_LEN})')
print(f'Output dir: {OUTPUT_DIR}')
print()

frame_lengths = []  # Track original lengths for stats
n_errors = 0

for i, wav_path in enumerate(wav_files):
    try:
        emb = extract_embedding(wav_path, model, max_len=MAX_LEN)
        out_name = wav_path.stem + '.npy'
        np.save(OUTPUT_DIR / out_name, emb)

        # Track original frame count (before padding)
        waveform, sr = torchaudio.load(wav_path)
        if sr != MODEL_SAMPLE_RATE:
            n_samples = int(waveform.shape[1] * MODEL_SAMPLE_RATE / sr)
        else:
            n_samples = waveform.shape[1]
        # wav2vec2 produces ~1 frame per 320 samples (20ms at 16kHz)
        est_frames = n_samples // 320
        frame_lengths.append(est_frames)

    except Exception as e:
        print(f'  ERROR on {wav_path.name}: {e}')
        n_errors += 1
        continue

    if (i + 1) % 100 == 0 or (i + 1) == len(wav_files):
        print(f'  [{i+1:>4}/{len(wav_files)}] {wav_path.name} → {out_name}  '
              f'{emb.shape}  (orig ~{est_frames} frames)')

print(f'\nDone. {len(wav_files) - n_errors} files written, {n_errors} errors.')
print(f'Frame length stats (before padding): '
      f'mean={np.mean(frame_lengths):.0f}, '
      f'max={np.max(frame_lengths)}, '
      f'p95={np.percentile(frame_lengths, 95):.0f}, '
      f'clipped to {MAX_LEN}')

Extracting wav2vec2-base embeddings...
Output shape per clip: (768, 230)
Output dir: /content/pretrained_embeddings

  [ 100/1009] audio_0099.wav → audio_0099.npy  (768, 230)  (orig ~73 frames)
  [ 200/1009] audio_0199.wav → audio_0199.npy  (768, 230)  (orig ~79 frames)
  [ 300/1009] audio_0299.wav → audio_0299.npy  (768, 230)  (orig ~72 frames)
  [ 400/1009] audio_0399.wav → audio_0399.npy  (768, 230)  (orig ~87 frames)
  [ 500/1009] audio_0499.wav → audio_0499.npy  (768, 230)  (orig ~51 frames)
  [ 600/1009] audio_0599.wav → audio_0599.npy  (768, 230)  (orig ~44 frames)
  [ 700/1009] audio_0699.wav → audio_0699.npy  (768, 230)  (orig ~45 frames)
  [ 800/1009] audio_0799.wav → audio_0799.npy  (768, 230)  (orig ~35 frames)
  [ 900/1009] audio_0899.wav → audio_0899.npy  (768, 230)  (orig ~62 frames)
  [1000/1009] audio_0999.wav → audio_0999.npy  (768, 230)  (orig ~216 frames)
  [1009/1009] audio_1008.wav → audio_1008.npy  (768, 230)  (orig ~52 frames)

Done. 1009 files written, 0 errors

## 5. Verify embeddings

In [15]:
# Quick sanity check
npy_files = sorted(OUTPUT_DIR.glob('*.npy'))
print(f'Total .npy files: {len(npy_files)}')

# Check a few
for f in npy_files[:3]:
    arr = np.load(f)
    print(f'  {f.name}: shape={arr.shape}, dtype={arr.dtype}, '
          f'mean={arr.mean():.4f}, std={arr.std():.4f}, '
          f'min={arr.min():.4f}, max={arr.max():.4f}')

# Verify all shapes are consistent
shapes = set()
for f in npy_files:
    shapes.add(np.load(f).shape)
print(f'\nUnique shapes: {shapes}')
assert len(shapes) == 1, f'Inconsistent shapes! {shapes}'
assert shapes.pop() == (768, MAX_LEN), 'Wrong shape!'
print('All shapes are (768, 230) ✓')

Total .npy files: 1009
  audio_0000.npy: shape=(768, 230), dtype=float32, mean=0.0000, std=0.1443, min=-3.8172, max=5.0430
  audio_0001.npy: shape=(768, 230), dtype=float32, mean=0.0001, std=0.1504, min=-3.7043, max=4.2470
  audio_0002.npy: shape=(768, 230), dtype=float32, mean=0.0003, std=0.1362, min=-4.0883, max=5.0462

Unique shapes: {(768, 230)}
All shapes are (768, 230) ✓


## 6. Download embeddings

Downloads a zip file containing all `.npy` embeddings.

**On RenkuLab**, unzip into:
```
/home/renku/work/data/pretrained_embeddings/
  audio_0000.npy
  audio_0001.npy
  ...
  audio_1008.npy
```

Then set `USE_PRETRAINED = True` in `src/config.py` and run the same training commands.

In [16]:
# Create zip
zip_out = '/content/pretrained_embeddings.zip'
with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUTPUT_DIR.glob('*.npy')):
        z.write(f, f.name)  # flat structure, no subfolder

zip_size = os.path.getsize(zip_out) / 1e6
print(f'Created: {zip_out} ({zip_size:.1f} MB)')
print(f'Contains {len(npy_files)} .npy files')

# Download
files.download(zip_out)
print('Download started — save to your machine, then upload to RenkuLab.')

Created: /content/pretrained_embeddings.zip (189.8 MB)
Contains 1009 .npy files


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started — save to your machine, then upload to RenkuLab.


## 7. What to do on next

After uploading and unzipping the embeddings:
In `src/config.py`, set:
```python
USE_PRETRAINED = True
USE_MFCC = False  # make sure this is off
```

Run the same training commands:
```bash
python scripts/train_multiseed.py --model audio --seeds 42 --use_existing_splits \
    --d_model 64 --num_layers 2 --dropout 0.5 --lr 0.001 --batch_size 32
```

The pipeline will automatically:
- Load `.npy` files from `pretrained_embeddings/` instead of spectrograms
- Set `F_DIM=768` (wav2vec2 embedding dimension)
- Adjust `input_proj` to `768 → d_model`
- Everything else stays the same